# Testing AMPKAR quantities of interest: Mass-action kinetics, double binding 

Nathaniel Linden (UCSD MAE) - May  2023

In [2]:
import numpy as np
from SALib.sample import sobol as sobol_samp
from SALib.sample import morris as morris_samp
from SALib.analyze import sobol as sobol_analyze
from SALib.analyze import morris as morris_analyze
from SALib.analyze.hdmr import analyze as hdmr_analyze
import os
import sys

import jax  
import jax.numpy as jnp
import jax.scipy as jsp
import scipy.optimize as opt
from jax import lax
import equinox as eqx
import diffrax as dfrx

import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from scipy.stats import t
import ipywidgets as widgets

jax.config.update('jax_enable_x64', True)
# load code
# sys.path.insert(0, '/scratch/nlinden/AMPK/src/ampk_models/odes')
# sys.path.insert(0, '/scratch/nlinden/AMPK/src/ampk_models/global_sensitivity_analysis')
sys.path.insert(0, '../odes')
import ampk_MA_double_mech_diffrax as model
import ampk_MA_double_mech as model1
from gsa_utils import *
import gsa_utils

%matplotlib inline
plt.style.use('~/.matplotlib/custom.mplstyle')
mpl.rcParams['figure.autolayout'] = True

### Load the parameter values
Here, we define two sets of bounds on the parameters. The first uses a $\pm 50 \%$ interval, while the second uses more extreme upper and lower bounds for some of the parameters. We chose the more extreme bounds for these parameters to ensure that we are covering all possible physiological values for these parameters.

In [3]:
MA_nominals = pd.read_csv('nominal_params_MA_tmp.csv')
MM_nominals = pd.read_csv('nominal_params_MM_tmp.csv')
nominal_vals_MA = MA_nominals['value'].to_list()
param_names_MA = MA_nominals['parameter'].to_list()
nominal_vals_MM = MM_nominals['value'].to_list()
param_names_MM = MM_nominals['parameter'].to_list()

# metabolism_params
metab_parms_basal = {'kGly': 1300.0,'kHydro':1.4e-3,'kForAK':40.44,
                     'kRevAK':1.1e-3,'VmaxOxPhos':0.5,'Kadp': 5.8e-2,'n': 2.568,}
metab_parms_stress = {'kGly': 4.5,'kHydro':1.4e-3,'kForAK':40.44,
                      'kRevAK':1.1e-3,'VmaxOxPhos':0.5,'Kadp': 5.8e-2,'n': 2.568,}

# bounds and problem dicts for SALib
lb_mult = 0.5
ub_mult = 1.5
bounds_MA_0 = [[lb_mult*param, ub_mult*param] for param in nominal_vals_MA]
bounds_MM_0 = [[lb_mult*param, ub_mult*param] for param in nominal_vals_MM]

MA_nominals = pd.read_csv('nominal_params_MA_tmp.csv')
MA_bounds = pd.read_csv('param_bounds_MA_tmp.csv')
MM_nominals = pd.read_csv('nominal_params_MM_tmp.csv')
nominal_vals_MA = MA_nominals['value'].to_list()
param_names_MA = MA_nominals['parameter'].to_list()
bounds_MA_1 = [MA_bounds['lower'].to_list(), MA_bounds['upper'].to_list()]
bounds_MA_1 = [[lb, ub] for lb, ub in zip(bounds_MA_1[0], bounds_MA_1[1])]

Set up the necesary overhead for solving the model.

In [51]:
state_names = ['AMP', 'ADP', 'ATP', 'AMPK', 'pAMPK', 'AMP_AMPK', 'ADP_AMPK', 'ATP_AMPK', 'AMP_pAMPK', 'ADP_pAMPK', 'ATP_pAMPK', 'AMP_AMP_AMPK', 'AMP_ADP_AMPK', 'AMP_ATP_AMPK', 'ADP_ADP_AMPK', 'ADP_ATP_AMPK', 'ATP_ATP_AMPK', 'AMP_AMP_pAMPK', 'AMP_ADP_pAMPK', 'AMP_ATP_pAMPK', 'ADP_ADP_pAMPK', 'ADP_ATP_pAMPK', 'ATP_ATP_pAMPK', 'CaMKK', 'CaMKK_AMPK', 'CaMKK_AMP_AMPK', 'CaMKK_ADP_AMPK', 'CaMKK_ATP_AMPK', 'CaMKK_AMP_AMP_AMPK', 'CaMKK_AMP_ADP_AMPK', 'CaMKK_AMP_ATP_AMPK', 'CaMKK_ADP_ADP_AMPK', 'CaMKK_ADP_ATP_AMPK', 'CaMKK_ATP_ATP_AMPK', 'LKB1', 'LKB1_AMP_AMPK', 'LKB1_ADP_AMPK', 'LKB1_AMP_AMP_AMPK', 'LKB1_AMP_ADP_AMPK', 'LKB1_ADP_ADP_AMPK', 'PP', 'PP_pAMPK', 'PP_ATP_pAMPK', 'PP_AMP_ATP_pAMPK', 'PP_ADP_ATP_pAMPK', 'PP_ATP_ATP_pAMPK', 'AMPKAR', 'pAMPKAR', 'AMPKAR_AMP_pAMPK', 'AMPKAR_AMP_AMP_pAMPK', 'AMPKAR_AMP_ADP_pAMPK', 'PP1', 'PP1_pAMPKAR']

# initial conditions
to_set = ['AMP', 'ADP', 'ATP', 'AMPK', 'CaMKK', 'LKB1', 'PP', 'AMPKAR', 'PP1']
idxs = [state_names.index(item) for item in to_set]

# print(idxs)

y0 = np.zeros((53,))
y0[idxs[0]] = 2e-5   # 'AMP' mM
y0[idxs[1]] = 1.3e-1 # 'ADP mM
y0[idxs[2]] = 8.2   # 'ATP mM
y0[idxs[3]] = 0.6   # 'AMPK mM
y0[idxs[4]] = 1.0   # 'CaMKK mM
y0[idxs[5]] = 1.0   # 'LKB1 mM
y0[idxs[6]] = 1.0   # 'PP mM
y0[idxs[7]] = 0.1   # 'AMPKAR mM
y0[idxs[8]] = 1.0   # 'PP1 mM

y0 = np.array(y0)

# function to compute all MA params from sampled params
def compute_MA_params(params):
    return (1.0, # kOnAMP
            params[0], # kOffAMP
            1.0 , # kOnADP
            params[1], # kOffADP
            1.0, # kOnATP
            params[2], # kOffATP
            params[3], # kOnCaMKK
            1.0, # kOffCaMKK
            params[4], # kPhosCaMKK
            params[5], # kOnLKB1
            1.0, # kOffLKB1
            params[6], # kPhosLKB1
            params[7], # kOnPP
            1.0, # kOffPP
            params[8], # kDephosPP
            params[9], # kOnAMPK
            1.0, # kOffAMPK
            params[10], # kPhosAMPK
            params[11], # kOnPP1
            1.0, # kOffPP1
            params[12]) # kDephosPP1

# parameters
# metabolism_params
metab_parms_basal = {'kGly': 1300.0,'kHydro':1.4e-3,'kForAK':40.44,
                     'kRevAK':1.1e-3,'VmaxOxPhos':0.5,'Kadp': 5.8e-2,'n': 2.568,}
metab_parms_stress = {'kGly': 4.5,'kHydro':1.4e-3,'kForAK':40.44,
                      'kRevAK':1.1e-3,'VmaxOxPhos':0.5,'Kadp': 5.8e-2,'n': 2.568,}

## Now test a solve to steady-state function
@jax.jit
def solve_model(params, rhs, y0, t1, times):
    y0 = y0.at[46].set(params[-1])
    params = compute_MA_params(params)
    solver=dfrx.Kvaerno5()
    stepsize_controller = dfrx.PIDController(rtol=1e-10, atol=1e-10)
    t0 = 0.0
    # times = jnp.arange(t0, t1, 0.5)
    dt0 = 1e-8 # initial time step
    saveat=dfrx.SaveAt(ts=times)

    # initial solve
    sol = dfrx.diffeqsolve(
        rhs, 
        solver, 
        t0, t1, dt0, 
        y0, 
        saveat=saveat, 
        stepsize_controller=stepsize_controller,
        args=params)
    
    return sol # returns the final state

################################################
#                   Model RHS                  #
################################################
rhs = model.ampk_MA_double_mech(**metab_parms_basal)
rhs_stress = model.ampk_MA_double_mech(**metab_parms_stress)
rhs = dfrx.ODETerm(rhs)
rhs_stress = dfrx.ODETerm(rhs_stress)

# run once to compile
times = jnp.arange(0.0, 1000.0, 0.5)
sol = solve_model(nominal_vals_MA, rhs_stress, y0, 1000.0, times)

Define a function that takes all of the variable model parameters as arguments, along with the QoI and plots the quantities of interest.

In [120]:
def update_plot(KdAMP, KdADP, KdATP, kOnCaMKK, kPhosCaMKK, kOnLKB1, kPhosLKB1, kOnPP, kDephosPP, kOnAMPK, kPhosAMPK, kOnPP1, kDephosPP1, AMPKAR_0, qoi_func):
    sample = np.array([KdAMP, KdADP, KdATP, kOnCaMKK, kPhosCaMKK, kOnLKB1, kPhosLKB1, kOnPP, kDephosPP, kOnAMPK*100, kPhosAMPK, kOnPP1, kDephosPP1, AMPKAR_0])
    
    # run the model to initial steady-state
    tmax = 1e5
    times = jnp.arange(0.0, tmax, tmax/1000)
    sol_basal = solve_model(sample, rhs, y0, tmax, times)
    # now actually run to get a solution
    # note sol[0] will be the ic and sol[1] will be the time to steady-state
    times = jnp.arange(0.0, tmax, tmax/1000)
    sol_stress = solve_model(sample, rhs_stress, sol_basal.ys[-1,:], tmax, times)

    # compute the QoI
    qoi_basal, _ = qoi_func(sol_basal)
    qoi_stress, qoi_name = qoi_func(sol_stress)

    # ploting stuff
    fig, ax = plt.subplots(2, 1, figsize=(6, 4))
    # solutions
    # AMPKAR_tot_basal = sol_basal.ys[:,46]+sol_basal.ys[:,47]+sol_basal.ys[:,48]+sol_basal.ys[:,49]+sol_basal.ys[:,50]+sol_basal.ys[:,52]
    # AMPKAR_tot_stress = sol_stress.ys[:,46]+sol_stress.ys[:,47]+sol_stress.ys[:,48]+sol_stress.ys[:,49]+sol_stress.ys[:,50]+sol_stress.ys[:,52]
    # ax[0].plot(sol_basal.ts, sol_basal.ys[:,47]/AMPKAR_tot_basal, 'k')
    # ax[0].plot(sol_basal.ts[-1]+sol_stress.ts, sol_stress.ys[:,47]/AMPKAR_tot_stress, 'b')
    # ax[0].set_ylim([0, 1e-7])
    # ax[0].set_ylabel(r'$pAMPKAR/AMPKAR_{tot}$')
    # ax[0].set_xlabel('time (s)')
    # ax[0].set_xlim([0.0, 2*tmax])

    # # qoi
    # ax[1].plot(sol_basal.ts, qoi_basal, 'k')
    # ax[1].plot(sol_basal.ts[-1]+sol_stress.ts, qoi_stress, 'b')
    # ax[1].set_ylabel(qoi_name)
    # # ax[1].set_ylim([0.0, 1e-7])
    # ax[1].set_xlabel('time (s)')
    # ax[1].set_xlim([0.0, 2*tmax])
    ampk_names = ['AMPK', 'pAMPK', 'AMP_AMPK', 'ADP_AMPK', 'ATP_AMPK', 'AMP_pAMPK', 'ADP_pAMPK', 'ATP_pAMPK', 'AMP_AMP_AMPK', 'AMP_ADP_AMPK', 'AMP_ATP_AMPK', 'ADP_ADP_AMPK', 'ADP_ATP_AMPK', 'ATP_ATP_AMPK', 'AMP_AMP_pAMPK', 'AMP_ADP_pAMPK', 'AMP_ATP_pAMPK', 'ADP_ADP_pAMPK', 'ADP_ATP_pAMPK', 'ATP_ATP_pAMPK','CaMKK_AMPK', 'CaMKK_AMP_AMPK', 'CaMKK_ADP_AMPK', 'CaMKK_ATP_AMPK', 'CaMKK_AMP_AMP_AMPK', 'CaMKK_AMP_ADP_AMPK', 'CaMKK_AMP_ATP_AMPK', 'CaMKK_ADP_ADP_AMPK', 'CaMKK_ADP_ATP_AMPK', 'CaMKK_ATP_ATP_AMPK', 'LKB1_AMP_AMPK', 'LKB1_ADP_AMPK', 'LKB1_AMP_AMP_AMPK', 'LKB1_AMP_ADP_AMPK', 'LKB1_ADP_ADP_AMPK', 'PP_pAMPK', 'PP_ATP_pAMPK', 'PP_AMP_ATP_pAMPK', 'PP_ADP_ATP_pAMPK', 'PP_ATP_ATP_pAMPK', 'AMPKAR_AMP_pAMPK', 'AMPKAR_AMP_AMP_pAMPK', 'AMPKAR_AMP_ADP_pAMPK']
    idxs = [state_names.index(item) for item in ampk_names]

    AMPK_tot_basal = sol_basal.ys[:,idxs[0]]
    AMPK_tot_stress = sol_stress.ys[:,idxs[0]]
    for idx in idxs[1:]:
        AMPK_tot_basal += sol_basal.ys[:,idx]
        AMPK_tot_stress += sol_stress.ys[:,idx]

    pampk_names = ['pAMPK','AMP_pAMPK','ADP_pAMPK','ATP_pAMPK', 'AMP_AMP_pAMPK', 'AMP_ADP_pAMPK', 'AMP_ATP_pAMPK', 'ADP_ADP_pAMPK', 'ADP_ATP_pAMPK', 'ATP_ATP_pAMPK','PP_pAMPK', 'PP_ATP_pAMPK', 'PP_AMP_ATP_pAMPK', 'PP_ADP_ATP_pAMPK', 'PP_ATP_ATP_pAMPK', 'AMPKAR_AMP_pAMPK', 'AMPKAR_AMP_AMP_pAMPK', 'AMPKAR_AMP_ADP_pAMPK']
    idxs = [state_names.index(item) for item in pampk_names]
    pAMPK_tot_basal = sol_basal.ys[:,idxs[0]]
    pAMPK_tot_stress = sol_stress.ys[:,idxs[0]]
    for idx in idxs[1:]:
        pAMPK_tot_basal += sol_basal.ys[:,idx]
        pAMPK_tot_stress += sol_stress.ys[:,idx]
  
    # pAMPKAR
    # AMPKAR_AMP_pAMPK + AMPKAR_AMP_AMP_pAMPK + AMPKAR_AMP_ADP_pAMPK
    ax[0].plot(sol_basal.ts[10:], pAMPK_tot_basal[10:], 'k')
    ax[0].plot(sol_basal.ts[-1]+sol_stress.ts, pAMPK_tot_stress, 'b')
    ax[0].set_ylim([0.5, 0.6001])

    # AMPKAR_AMP_pAMPK + AMPKAR_AMP_AMP_pAMPK + AMPKAR_AMP_ADP_pAMPK
    ax[1].plot(sol_basal.ts, AMPK_tot_basal, 'k')
    ax[1].plot(sol_basal.ts[-1]+sol_stress.ts, AMPK_tot_stress, 'b')
    ax[1].set_ylim([0, 1e0])

    plt.show()

Define QoI functions

In [121]:
def ratio(sol):
    ampk_tot = sol.ys[:,46]+sol.ys[:,47]+sol.ys[:,48]+sol.ys[:,49]+sol.ys[:,50]+sol.ys[:,52]
    pampk = sol.ys[:,47]

    ratio = (pampk/ampk_tot)
    return ratio, r"$pAMPKAR/AMPKAR_{tot}$"

def change(sol):
    ampk_tot = sol.ys[:,46]+sol.ys[:,47]+sol.ys[:,48]+sol.ys[:,49]+sol.ys[:,50]+sol.ys[:,52]
    pampk = sol.ys[:,47]

    change = (pampk/ampk_tot) - (pampk[0]/ampk_tot[0])
    return change, r"change $pAMPKAR/AMPKAR_{tot}$"

def normalized_change(sol):
    ampk_tot = sol.ys[:,46]+sol.ys[:,47]+sol.ys[:,48]+sol.ys[:,49]+sol.ys[:,50]+sol.ys[:,52]
    pampk = sol.ys[:,47]

    change = ((pampk/ampk_tot) - (pampk[0]/ampk_tot[0]))/(pampk[0]/ampk_tot[0])
    return change, r"norm change $pAMPKAR/AMPKAR_{tot}$"

In [122]:
KdAMP_slider = widgets.FloatSlider(min=bounds_MA_0[0][0], max=bounds_MA_0[0][1], 
                                      value=nominal_vals_MA[0], 
                                      description=param_names_MA[0],
                                      step=1e-8, continous_update=True, 
                                      readout_format='.8f')
KdADP_slider = widgets.FloatSlider(min=bounds_MA_0[1][0], max=bounds_MA_0[1][1], 
                                      value=nominal_vals_MA[1], 
                                      description=param_names_MA[1],
                                      step=1e-8, continous_update=True,
                                      readout_format='.8f')
KdATP_slider = widgets.FloatSlider(min=bounds_MA_0[2][0], max=100*bounds_MA_0[2][1], 
                                      value=nominal_vals_MA[2], 
                                      description=param_names_MA[2],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnCaMKK_slider = widgets.FloatSlider(min=bounds_MA_0[3][0], max=bounds_MA_0[3][1], 
                                      value=nominal_vals_MA[3], 
                                      description=param_names_MA[3],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosCaMKK_slider = widgets.FloatSlider(min=bounds_MA_0[4][0], max=bounds_MA_0[4][1], 
                                      value=nominal_vals_MA[4], 
                                      description=param_names_MA[4],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnLKB1_slider = widgets.FloatSlider(min=bounds_MA_0[5][0], max=bounds_MA_0[5][1], 
                                      value=nominal_vals_MA[5], 
                                      description=param_names_MA[5],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosLKB1_slider = widgets.FloatSlider(min=bounds_MA_0[6][0], max=bounds_MA_0[6][1], 
                                      value=nominal_vals_MA[6], 
                                      description=param_names_MA[6],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnPP_slider = widgets.FloatSlider(min=bounds_MA_0[7][0], max=bounds_MA_0[7][1], 
                                      value=nominal_vals_MA[7], 
                                      description=param_names_MA[7],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kDephosPP_slider = widgets.FloatSlider(min=bounds_MA_0[8][0], max=bounds_MA_0[8][1], 
                                      value=nominal_vals_MA[8], 
                                      description=param_names_MA[8],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnAMPK_slider = widgets.FloatSlider(min=bounds_MA_0[9][0], max=bounds_MA_0[9][1], 
                                      value=nominal_vals_MA[9], 
                                      description=param_names_MA[9],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosAMPK_slider = widgets.FloatSlider(min=bounds_MA_0[10][0], max=bounds_MA_0[10][1], 
                                      value=nominal_vals_MA[10], 
                                      description=param_names_MA[10],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnPP1_slider = widgets.FloatSlider(min=bounds_MA_0[11][0], max=bounds_MA_0[11][1], 
                                      value=nominal_vals_MA[11], 
                                      description=param_names_MA[11],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kDephosPP1_slider = widgets.FloatSlider(min=bounds_MA_0[12][0], max=bounds_MA_0[12][1], 
                                      value=nominal_vals_MA[12], 
                                      description=param_names_MA[12],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
AMPKAR_0_slider = widgets.FloatSlider(min=bounds_MA_0[13][0], max=1000*bounds_MA_0[13][1], 
                                      value=nominal_vals_MA[13], 
                                      description=param_names_MA[13],
                                      step=1e-8, continous_update=True, 
                                      readout_format='.4f')

widget = widgets.interactive(update_plot, KdAMP=KdAMP_slider, KdADP=KdADP_slider, 
                             KdATP=KdATP_slider, kOnCaMKK=kOnCaMKK_slider, kPhosCaMKK=kPhosCaMKK_slider,
                             kOnLKB1=kOnLKB1_slider, kPhosLKB1=kPhosLKB1_slider, 
                             kOnPP=kOnPP_slider, kDephosPP=kDephosPP_slider,
                             kOnAMPK=kOnAMPK_slider, kPhosAMPK=kPhosAMPK_slider,
                             kOnPP1=kOnPP1_slider, kDephosPP1=kDephosPP1_slider,
                             AMPKAR_0=AMPKAR_0_slider, qoi_func=widgets.fixed(change))
display(widget)


interactive(children=(FloatSlider(value=0.0025, description='KdAMP', max=0.00375, min=0.00125, readout_format=…

In [33]:
KdAMP_slider = widgets.FloatSlider(min=bounds_MA_1[0][0], max=bounds_MA_1[0][1], 
                                      value=nominal_vals_MA[0], 
                                      description=param_names_MA[0],
                                      step=1e-8, continous_update=True, 
                                      readout_format='.8f')
KdADP_slider = widgets.FloatSlider(min=bounds_MA_1[1][0], max=bounds_MA_1[1][1], 
                                      value=nominal_vals_MA[1], 
                                      description=param_names_MA[1],
                                      step=1e-8, continous_update=True,
                                      readout_format='.8f')
KdATP_slider = widgets.FloatSlider(min=bounds_MA_1[2][0], max=bounds_MA_1[2][1], 
                                      value=nominal_vals_MA[2], 
                                      description=param_names_MA[2],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnCaMKK_slider = widgets.FloatSlider(min=bounds_MA_1[3][0], max=bounds_MA_1[3][1], 
                                      value=nominal_vals_MA[3], 
                                      description=param_names_MA[3],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosCaMKK_slider = widgets.FloatSlider(min=bounds_MA_1[4][0], max=bounds_MA_1[4][1], 
                                      value=nominal_vals_MA[4], 
                                      description=param_names_MA[4],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnLKB1_slider = widgets.FloatSlider(min=bounds_MA_1[5][0], max=bounds_MA_1[5][1], 
                                      value=nominal_vals_MA[5], 
                                      description=param_names_MA[5],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosLKB1_slider = widgets.FloatSlider(min=bounds_MA_1[6][0], max=bounds_MA_1[6][1], 
                                      value=nominal_vals_MA[6], 
                                      description=param_names_MA[6],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnPP_slider = widgets.FloatSlider(min=bounds_MA_1[7][0], max=bounds_MA_1[7][1], 
                                      value=nominal_vals_MA[7], 
                                      description=param_names_MA[7],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kDephosPP_slider = widgets.FloatSlider(min=bounds_MA_1[8][0], max=bounds_MA_1[8][1], 
                                      value=nominal_vals_MA[8], 
                                      description=param_names_MA[8],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnAMPK_slider = widgets.FloatSlider(min=bounds_MA_1[9][0], max=bounds_MA_1[9][1], 
                                      value=nominal_vals_MA[9], 
                                      description=param_names_MA[9],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kPhosAMPK_slider = widgets.FloatSlider(min=bounds_MA_1[10][0], max=bounds_MA_1[10][1], 
                                      value=nominal_vals_MA[10], 
                                      description=param_names_MA[10],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kOnPP1_slider = widgets.FloatSlider(min=bounds_MA_1[11][0], max=bounds_MA_1[11][1], 
                                      value=nominal_vals_MA[11], 
                                      description=param_names_MA[11],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
kDephosPP1_slider = widgets.FloatSlider(min=bounds_MA_1[12][0], max=bounds_MA_1[12][1], 
                                      value=nominal_vals_MA[12], 
                                      description=param_names_MA[12],
                                      step=1e-8, continous_update=True,
                                      readout_format='.4f')
AMPKAR_0_slider = widgets.FloatSlider(min=bounds_MA_1[13][0], max=100*bounds_MA_1[13][1], 
                                      value=nominal_vals_MA[13], 
                                      description=param_names_MA[13],
                                      step=1e-8, continous_update=True, 
                                      readout_format='.4f')

widget = widgets.interactive(update_plot, KdAMP=KdAMP_slider, KdADP=KdADP_slider, 
                             KdATP=KdATP_slider, kOnCaMKK=kOnCaMKK_slider, kPhosCaMKK=kPhosCaMKK_slider,
                             kOnLKB1=kOnLKB1_slider, kPhosLKB1=kPhosLKB1_slider, 
                             kOnPP=kOnPP_slider, kDephosPP=kDephosPP_slider,
                             kOnAMPK=kOnAMPK_slider, kPhosAMPK=kPhosAMPK_slider,
                             kOnPP1=kOnPP1_slider, kDephosPP1=kDephosPP1_slider,
                             AMPKAR_0=AMPKAR_0_slider, qoi_func=widgets.fixed(ratio))
display(widget)


interactive(children=(FloatSlider(value=0.0025, description='KdAMP', max=0.00375, min=0.00125, readout_format=…